In [ ]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / "data" / "claims.csv").exists():
    root = root.parent

claims_path = root / "data" / "claims.csv"
plans_path = root / "data" / "plans.csv"

claims_df = pd.read_csv(claims_path)
plans_df = pd.read_csv(plans_path)

for name, df in [("claims", claims_df), ("plans", plans_df)]:
    print(f"\n{name.upper()} INFO")
    df.info()
    print(f"\n{name.upper()} HEAD")
    display(df.head())

claims_df = claims_df.drop_duplicates()
plans_df = plans_df.drop_duplicates()

for df in (claims_df, plans_df):
    df.dropna(how="all", inplace=True)
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]):
            df[col] = df[col].fillna("")
        else:
            df[col] = df[col].fillna(0)

if "date_filed" in claims_df.columns:
    claims_df["date_filed"] = pd.to_datetime(claims_df["date_filed"], errors="coerce")

if "date_filed" in plans_df.columns:
    plans_df["date_filed"] = pd.to_datetime(plans_df["date_filed"], errors="coerce")

claims_df.head(), plans_df.head()

In [ ]:
import sqlite3
from pathlib import Path

output_db = root / "coverage.db"
conn = sqlite3.connect(output_db)

claims_df.to_sql("claims", conn, if_exists="replace", index=False)
plans_df.to_sql("plans", conn, if_exists="replace", index=False)

conn.commit()
conn.close()

print(f"Database created at: {output_db}")

In [3]:
import sqlite3
from pathlib import Path
import pandas as pd

# Use the explicit workspace path to ensure the notebook can find the CSV files
root = Path(r"C:/Users/Jidnyasa Patil/Downloads/AI-cohort")
claims_path = root / "data" / "claims.csv"
plans_path = root / "data" / "plans.csv"

claims_df = pd.read_csv(claims_path)
plans_df = pd.read_csv(plans_path)

output_db = root / "coverage.db"
conn = sqlite3.connect(output_db)

claims_df.to_sql("claims", conn, if_exists="replace", index=False)
plans_df.to_sql("plans", conn, if_exists="replace", index=False)

conn.commit()
conn.close()

print(f"Database created at: {output_db}")

query_examples = {
    "deductible_on_gold_ppo": """
        SELECT annual_deductible
        FROM plans
        WHERE plan_name = 'Gold PPO';
    """,
    "pending_claims_for_member": """
        SELECT COUNT(*) AS pending_claim_count
        FROM claims
        WHERE member_id = 'M1001' AND status = 'Pending';
    """,
    "plans_under_400": """
        SELECT plan_id, plan_name, monthly_premium
        FROM plans
        WHERE monthly_premium < 400
        ORDER BY monthly_premium;
    """,
    "join_claims_plans": """
        SELECT c.claim_id, c.member_id, p.plan_name, c.procedure, c.status
        FROM claims AS c
        JOIN plans AS p ON c.plan_id = p.plan_id;
    """,
    "top_procedures": """
        SELECT procedure, COUNT(*) AS claim_count
        FROM claims
        GROUP BY procedure
        ORDER BY claim_count DESC, procedure
        LIMIT 5;
    """
}

conn = sqlite3.connect(output_db)
for name, query in query_examples.items():
    print(f"\n-- {name}")
    for row in conn.execute(query):
        print(row)
conn.close()


EmptyDataError: No columns to parse from file